In [ ]:
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(), override=True)  
from langchain_groq import ChatGroq
chatModel = ChatGroq(
    model="llama-3.3-70b-versatile"
)

### RunnablePassthrough
-it doesnot do anything to the input data


-a chain with just RunnablePassthrough() will output the original input without any modification.

In [2]:
from langchain_core.runnables import RunnablePassthrough
chain=RunnablePassthrough()

In [3]:
chain.invoke("Hello, how are you?")

'Hello, how are you?'

### RunnableLambda
-To use a custom function inside a LCEL chain we need to wrap it up with RunnableLambda.

In [4]:
def russian_lastname(name:str)->str:
    return f"{name}ovich"

In [5]:
russian_lastname("Ivan")

'Ivanovich'

In [6]:
from langchain_core.runnables import RunnableLambda
chain=RunnablePassthrough() | RunnableLambda(russian_lastname) 

In [7]:
chain.invoke("Hello, how are you?")

'Hello, how are you?ovich'

### RunnableParallel

-We will use RunnableParallel() ofr running taks in parallel

-this is most important and most useful Runnable form Langchain

-In the following chain,RunnableParallel is going to run these two tasks in parallel

        -operation_a will be use RunnablePassthrough
        -operation_b will use RunnableLambda with russian_lastname function

In [8]:
from langchain_core.runnables import RunnableParallel
chain=RunnableParallel(
    {
        "operation_a":RunnablePassthrough(), 
        "operation_b":RunnableLambda(russian_lastname)
    })

In [9]:
chain.invoke("karthik")

{'operation_a': 'karthik', 'operation_b': 'karthikovich'}

In [11]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

In [22]:
prompt=ChatPromptTemplate.from_template("tell me a curious fact about {soccer_player}")
output_parser=StrOutputParser()

In [23]:
def russian_lastname_from_dictionary(person):
    return person["name"] + "ovich"

In [26]:
prompt = ChatPromptTemplate.from_template(
    """
    Original: {operation_a}

    Russian Name: {operation_b}

    Original Again: {operation_c}
    """
)

In [27]:
chain=RunnableParallel(
    {
        "operation_a": RunnablePassthrough(),
        "operation_b":RunnableLambda(russian_lastname_from_dictionary),
        "operation_c":RunnablePassthrough()
    }
)|prompt|chatModel|output_parser

In [30]:
chain.invoke({
    "anme1":"a fek",
    "name":"anirudh"
})

'It appears you\'ve provided a snippet of text that includes a dictionary with names, a mention of a Russian name, and then the dictionary again. Here\'s a breakdown and a possible interpretation of what you\'re presenting:\n\n1. **Original Dictionary**: You start with a dictionary that contains two key-value pairs. The keys are \'anme1\' and \'name\', with values \'a fek\' and \'anirudh\', respectively. This dictionary seems to have a typo in the first key (\'anme1\' instead of possibly \'name1\').\n\n2. **Russian Name**: You then mention \'anirudhovich\', which appears to be a Russian patronymic surname derived from \'Anirudh\'. In Russian culture, surnames are often formed by adding a suffix to the father\'s name, indicating "son of." The suffix \'-ovich\' is used for males, and \'-ovna\' or \'-ovna\' (depending on the region) for females. So, \'anirudhovich\' could be interpreted as "son of Anirudh."\n\n3. **Original Dictionary Again**: The dictionary is presented again, unchanged.

### More Advance RunnableParallel

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

vectorstore = FAISS.from_texts(
    ["dswithbappy focuses on providing content on Data Science, AI, ML, DL, CV, NLP, Python programming, etc. in English."], embedding=OpenAIEmbeddings()
)

retriever = vectorstore.as_retriever()

template = """Answer the question based only on the following context:
{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

model = ChatOpenAI(model="gpt-3.5-turbo")

retrieval_chain = (
    RunnableParallel({"context": retriever, "question": RunnablePassthrough()})
    | prompt
    | model
    | StrOutputParser()
)

retrieval_chain.invoke("What is flung?")

In [ ]:
from operator import itemgetter
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

model = ChatOpenAI(model="gpt-3.5-turbo")

vectorstore = FAISS.from_texts(
    ["dswithbappy focuses on providing content on Data Science, AI, ML, DL, CV, NLP, Python programming, etc. in English."], embedding=OpenAIEmbeddings()
)
retriever = vectorstore.as_retriever()

template = """Answer the question based only on the following context:
{context}

Question: {question}

Answer in the following language: {language}
"""

prompt = ChatPromptTemplate.from_template(template)

chain = (
    {
        "context": itemgetter("question") |  etriever,
        "question": itemgetter("question"),
        "language": itemgetter("language"),
    }
    | prompt
    | model
    | StrOutputParser()
)
chain.invoke({"question": "What is dswithbappy?", "language": "Pirate English"})